In [4]:
import sys
import subprocess

print("Python:", sys.version)
print("Pip:")
subprocess.run([sys.executable, "-m", "pip", "--version"])

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Pip:


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '--version'], returncode=0)

In [5]:
# Install only Python 3.13-compatible core dependencies

!pip install -q -U datasets transformers spacy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 74.2 MB/s eta 0:00:00


In [6]:
from datasets import load_dataset
import spacy
import transformers
import datasets

print("Python environment:")
print("spaCy:", spacy.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)

print("\nLoading CoNLL-2003...")

dataset = load_dataset("lhoestq/conll2003")

print("\nDataset:")
print(dataset)

print("\nFeatures:")
print(dataset["train"].features)

Python environment:
spaCy: 3.8.16
Transformers: 5.17.0
Datasets: 5.0.1

Loading CoNLL-2003...


dataset_infos.json:   0%|          | 0.00/3.67k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  281kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]


Dataset:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Features:
{'id': Value('string'), 'tokens': List(Value('string')), 'pos_tags': List(Value('int64')), 'chunk_tags': List(Value('int64')), 'ner_tags': List(Value('int64'))}


In [7]:
from collections import Counter

# CoNLL-2003 label mapping
label_names = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-MISC",
    "I-MISC",
]

# Count NER tags across the training set
train_ner_tags = [
    tag
    for example in dataset["train"]
    for tag in example["ner_tags"]
]

tag_counts = Counter(train_ner_tags)

print("NER Label Distribution")
print("=" * 40)

for tag_id, count in sorted(tag_counts.items()):
    print(f"{tag_id:>2}  {label_names[tag_id]:<8} {count:,}")

print("\nTotal token-level labels:", len(train_ner_tags))

# Count sentences
print("\nDataset Sizes")
print("=" * 40)
print(f"Train:      {len(dataset['train']):,}")
print(f"Validation: {len(dataset['validation']):,}")
print(f"Test:       {len(dataset['test']):,}")

# Show a real example
example = dataset["train"][0]

print("\nExample")
print("=" * 40)

for token, tag_id in zip(example["tokens"], example["ner_tags"]):
    print(f"{token:<20} {label_names[tag_id]}")

NER Label Distribution
 0  O        169,578
 1  B-PER    6,600
 2  I-PER    4,528
 3  B-ORG    6,321
 4  I-ORG    3,704
 5  B-LOC    7,140
 6  I-LOC    1,157
 7  B-MISC   3,438
 8  I-MISC   1,155

Total token-level labels: 203621

Dataset Sizes
Train:      14,041
Validation: 3,250
Test:       3,453

Example
EU                   B-ORG
rejects              O
German               B-MISC
call                 O
to                   O
boycott              O
British              B-MISC
lamb                 O
.                    O


In [8]:
from typing import List, Tuple


def iob_to_entities(tokens: List[str], tags: List[str]) -> List[Tuple[str, str, int, int]]:
    """
    Convert IOB2 tags into entity spans.

    Returns:
        List of tuples:
        (entity_text, entity_type, start_token, end_token)
    """
    entities = []

    current_tokens = []
    current_type = None
    start_idx = None

    def close_entity(end_idx):
        nonlocal current_tokens, current_type, start_idx

        if current_tokens:
            entity_text = " ".join(current_tokens)
            entities.append(
                (
                    entity_text,
                    current_type,
                    start_idx,
                    end_idx,
                )
            )

        current_tokens = []
        current_type = None
        start_idx = None

    for idx, (token, tag) in enumerate(zip(tokens, tags)):
        if tag == "O":
            close_entity(idx - 1)
            continue

        prefix, entity_type = tag.split("-", 1)

        if prefix == "B":
            close_entity(idx - 1)

            current_tokens = [token]
            current_type = entity_type
            start_idx = idx

        elif prefix == "I":
            if current_type == entity_type and current_tokens:
                current_tokens.append(token)
            else:
                # Handle malformed IOB sequences safely
                close_entity(idx - 1)

                current_tokens = [token]
                current_type = entity_type
                start_idx = idx

    close_entity(len(tokens) - 1)

    return entities


# Test on the first training example
example = dataset["train"][0]

tokens = example["tokens"]
tags = [label_names[tag_id] for tag_id in example["ner_tags"]]

entities = iob_to_entities(tokens, tags)

print("Extracted Entities")
print("=" * 50)

for entity_text, entity_type, start, end in entities:
    print(
        f"{entity_type:<6} | "
        f"{entity_text:<30} | "
        f"tokens {start}-{end}"
    )

print("\nTotal entities:", len(entities))

Extracted Entities
ORG    | EU                             | tokens 0-0
MISC   | German                         | tokens 2-2
MISC   | British                        | tokens 6-6

Total entities: 3


In [9]:
import spacy
from collections import Counter

MODEL_NAME = "en_core_web_sm"

try:
    nlp = spacy.load(MODEL_NAME)
except OSError:
    print(f"Downloading {MODEL_NAME}...")
    spacy.cli.download(MODEL_NAME)
    nlp = spacy.load(MODEL_NAME)

print("spaCy model:", MODEL_NAME)
print("spaCy version:", spacy.__version__)

print("\nPipeline components:")
print(nlp.pipe_names)

print("\nNER labels:")
print(nlp.get_pipe("ner").labels)

spaCy model: en_core_web_sm
spaCy version: 3.8.16

Pipeline components:
['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

NER labels:
('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


In [10]:
from collections import defaultdict
from sklearn.metrics import precision_recall_fscore_support

# Map spaCy entity labels to CoNLL-2003 labels
SPACY_TO_CONLL = {
    "PERSON": "PER",
    "ORG": "ORG",
    "GPE": "LOC",
    "LOC": "LOC",
    "MISC": "MISC",
}

TARGET_TYPES = ["PER", "ORG", "LOC", "MISC"]


def spacy_entities_from_tokens(tokens):
    """
    Run spaCy NER on a tokenized sentence while preserving
    the original token boundaries.
    """
    text = " ".join(tokens)
    doc = nlp(text)

    predictions = []

    for ent in doc.ents:
        entity_type = SPACY_TO_CONLL.get(ent.label_)

        if entity_type is not None:
            predictions.append(
                (
                    ent.text.strip(),
                    entity_type,
                )
            )

    return predictions


def normalize_entity(text):
    return " ".join(text.split()).strip()


# Evaluate a manageable first subset to verify the pipeline.
# We will run the complete test set after validation.
EVAL_SIZE = 500

gold_entities = []
pred_entities = []

for example in dataset["test"].select(range(EVAL_SIZE)):
    tokens = example["tokens"]

    gold_tags = [
        label_names[tag_id]
        for tag_id in example["ner_tags"]
    ]

    gold = iob_to_entities(tokens, gold_tags)

    gold_set = {
        (
            normalize_entity(entity_text),
            entity_type,
        )
        for entity_text, entity_type, _, _ in gold
    }

    predicted = spacy_entities_from_tokens(tokens)

    predicted_set = {
        (
            normalize_entity(entity_text),
            entity_type,
        )
        for entity_text, entity_type in predicted
    }

    gold_entities.append(gold_set)
    pred_entities.append(predicted_set)


# Entity-level exact-match evaluation
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

for gold_set, pred_set in zip(gold_entities, pred_entities):
    for entity in pred_set:
        entity_type = entity[1]

        if entity in gold_set:
            tp[entity_type] += 1
        else:
            fp[entity_type] += 1

    for entity in gold_set:
        entity_type = entity[1]

        if entity not in pred_set:
            fn[entity_type] += 1


print("spaCy Baseline — Entity-Level Evaluation")
print("=" * 60)

all_tp = all_fp = all_fn = 0

for entity_type in TARGET_TYPES:
    t = tp[entity_type]
    f_p = fp[entity_type]
    f_n = fn[entity_type]

    precision = t / (t + f_p) if (t + f_p) else 0.0
    recall = t / (t + f_n) if (t + f_n) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    all_tp += t
    all_fp += f_p
    all_fn += f_n

    print(
        f"{entity_type:<6} "
        f"P={precision:.4f}  "
        f"R={recall:.4f}  "
        f"F1={f1:.4f}"
    )

overall_precision = all_tp / (all_tp + all_fp)
overall_recall = all_tp / (all_tp + all_fn)
overall_f1 = (
    2 * overall_precision * overall_recall
    / (overall_precision + overall_recall)
)

print("\nOverall")
print("-" * 60)
print(f"Precision: {overall_precision:.4f}")
print(f"Recall:    {overall_recall:.4f}")
print(f"F1 Score:  {overall_f1:.4f}")
print(f"Evaluated samples: {EVAL_SIZE}")

spaCy Baseline — Entity-Level Evaluation
PER    P=0.8328  R=0.5973  F1=0.6957
ORG    P=0.3117  R=0.2553  F1=0.2807
LOC    P=0.7614  R=0.7230  F1=0.7417
MISC   P=0.0000  R=0.0000  F1=0.0000

Overall
------------------------------------------------------------
Precision: 0.6980
Recall:    0.5278
F1 Score:  0.6011
Evaluated samples: 500


In [11]:
# Full CoNLL-2003 test-set evaluation

EVAL_SIZE = len(dataset["test"])

gold_entities = []
pred_entities = []

print(f"Evaluating {EVAL_SIZE:,} test samples...")

for i, example in enumerate(dataset["test"]):
    tokens = example["tokens"]

    gold_tags = [
        label_names[tag_id]
        for tag_id in example["ner_tags"]
    ]

    gold = iob_to_entities(tokens, gold_tags)

    gold_set = {
        (
            normalize_entity(entity_text),
            entity_type,
        )
        for entity_text, entity_type, _, _ in gold
    }

    predicted = spacy_entities_from_tokens(tokens)

    predicted_set = {
        (
            normalize_entity(entity_text),
            entity_type,
        )
        for entity_text, entity_type in predicted
    }

    gold_entities.append(gold_set)
    pred_entities.append(predicted_set)

    if (i + 1) % 500 == 0:
        print(f"Processed: {i + 1:,}/{EVAL_SIZE:,}")


# Calculate metrics
tp = defaultdict(int)
fp = defaultdict(int)
fn = defaultdict(int)

for gold_set, pred_set in zip(gold_entities, pred_entities):

    for entity in pred_set:
        entity_type = entity[1]

        if entity in gold_set:
            tp[entity_type] += 1
        else:
            fp[entity_type] += 1

    for entity in gold_set:
        entity_type = entity[1]

        if entity not in pred_set:
            fn[entity_type] += 1


print("\nspaCy Baseline — Full CoNLL-2003 Test Set")
print("=" * 65)

all_tp = all_fp = all_fn = 0

results = {}

for entity_type in TARGET_TYPES:

    t = tp[entity_type]
    f_p = fp[entity_type]
    f_n = fn[entity_type]

    precision = t / (t + f_p) if (t + f_p) else 0.0
    recall = t / (t + f_n) if (t + f_n) else 0.0

    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )

    results[entity_type] = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": t,
        "fp": f_p,
        "fn": f_n,
    }

    all_tp += t
    all_fp += f_p
    all_fn += f_n

    print(
        f"{entity_type:<6} "
        f"P={precision:.4f}  "
        f"R={recall:.4f}  "
        f"F1={f1:.4f}"
    )


overall_precision = all_tp / (all_tp + all_fp)
overall_recall = all_tp / (all_tp + all_fn)

overall_f1 = (
    2 * overall_precision * overall_recall
    / (overall_precision + overall_recall)
)

print("\nOverall")
print("-" * 65)
print(f"Precision: {overall_precision:.4f}")
print(f"Recall:    {overall_recall:.4f}")
print(f"F1 Score:  {overall_f1:.4f}")
print(f"Test samples: {EVAL_SIZE:,}")

Evaluating 3,453 test samples...
Processed: 500/3,453
Processed: 1,000/3,453
Processed: 1,500/3,453
Processed: 2,000/3,453
Processed: 2,500/3,453
Processed: 3,000/3,453

spaCy Baseline — Full CoNLL-2003 Test Set
PER    P=0.7375  R=0.6146  F1=0.6705
ORG    P=0.4433  R=0.3081  F1=0.3635
LOC    P=0.7809  R=0.7330  F1=0.7562
MISC   P=0.0000  R=0.0000  F1=0.0000

Overall
-----------------------------------------------------------------
Precision: 0.6711
Recall:    0.4837
F1 Score:  0.5622
Test samples: 3,453


In [12]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )
else:
    print("WARNING: CUDA is not available.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [13]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

MODEL_CHECKPOINT = "distilbert/distilbert-base-cased"

label_names = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-MISC",
    "I-MISC",
]

label2id = {label: idx for idx, label in enumerate(label_names)}
id2label = {idx: label for idx, label in enumerate(label_names)}

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print("Tokenizer:", MODEL_CHECKPOINT)
print("Fast tokenizer:", tokenizer.is_fast)

print("\nLabels:")
for idx, label in enumerate(label_names):
    print(f"{idx}: {label}")

Loading tokenizer...
Tokenizer: distilbert/distilbert-base-cased
Fast tokenizer: True

Labels:
0: O
1: B-PER
2: I-PER
3: B-ORG
4: I-ORG
5: B-LOC
6: I-LOC
7: B-MISC
8: I-MISC


In [14]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=256,
        is_split_into_words=True,
    )

    all_labels = []

    for batch_index, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)

        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:
            # Special tokens
            if word_id is None:
                aligned_labels.append(-100)

            # First subword of a word
            elif word_id != previous_word_id:
                aligned_labels.append(labels[word_id])

            # Additional subwords
            else:
                # Continue the IOB entity label for subword pieces
                original_label = labels[word_id]

                if original_label in [1, 3, 5, 7]:
                    # B-X becomes I-X for subsequent subwords
                    aligned_labels.append(original_label + 1)
                else:
                    aligned_labels.append(original_label)

            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs


print("Tokenizing training data...")

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing CoNLL-2003",
)

print("\nTokenized dataset:")
print(tokenized_dataset)

print("\nFirst tokenized example:")
print(tokenized_dataset["train"][0].keys())

print("\nFirst example input IDs:")
print(tokenized_dataset["train"][0]["input_ids"][:20])

print("\nFirst example labels:")
print(tokenized_dataset["train"][0]["labels"][:20])

Tokenizing training data...


Tokenizing CoNLL-2003:   0%|          | 0/14041 [00:00<?, ? examples/s]

Tokenizing CoNLL-2003:   0%|          | 0/3250 [00:00<?, ? examples/s]

Tokenizing CoNLL-2003:   0%|          | 0/3453 [00:00<?, ? examples/s]


Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

First tokenized example:
dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])

First example input IDs:
[101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]

First example labels:
[-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]


In [16]:
# Fix Transformers version mismatch
!pip install -q --force-reinstall "transformers==4.57.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [17]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.17.0
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [18]:
# Remove the conflicting Transformers installation completely
!pip uninstall -y transformers

# Install a stable Transformers 4.x release
!pip install -q --no-cache-dir "transformers==4.57.3" --no-deps

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 158.5 MB/s eta 0:00:00


In [19]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.17.0
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [20]:
# Clean the Transformers package completely
!pip uninstall -y transformers

# Remove leftover package files
!rm -rf /usr/local/lib/python3.13/dist-packages/transformers
!rm -rf /usr/local/lib/python3.13/dist-packages/transformers-*.dist-info

# Install one clean Transformers version
!pip install -q --no-cache-dir "transformers==5.17.0"

Found existing installation: transformers 4.57.3
Uninstalling transformers-4.57.3:
  Successfully uninstalled transformers-4.57.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 287.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 398.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 258.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.0.1 requires fsspec[http]<=2026.6.0,>=2023.1.0, but you have fsspec 2026.9.0 which is incompatible.


In [1]:
import transformers
import torch

print("Transformers:", transformers.__version__)
print("Transformers path:", transformers.__file__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Transformers: 5.17.0
Transformers path: /usr/local/lib/python3.13/dist-packages/transformers/__init__.py
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [2]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

print("Transformers model imports: OK")

tokenizer_test = AutoTokenizer.from_pretrained(
    "distilbert/distilbert-base-cased"
)

model_test = AutoModelForTokenClassification.from_pretrained(
    "distilbert/distilbert-base-cased",
    num_labels=9
)

print("Tokenizer: OK")
print("Model: OK")
print("Model type:", model_test.__class__.__name__)

Transformers model imports: OK


model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer: OK
Model: OK
Model type: DistilBertForTokenClassification


In [4]:
# Rebuild CoNLL-2003 label mappings after Runtime Restart

label_names = [
    "O",
    "B-PER",
    "I-PER",
    "B-ORG",
    "I-ORG",
    "B-LOC",
    "I-LOC",
    "B-MISC",
    "I-MISC",
]

label2id = {
    label: index
    for index, label in enumerate(label_names)
}

id2label = {
    index: label
    for index, label in enumerate(label_names)
}

entity_types = [
    "PER",
    "ORG",
    "LOC",
    "MISC",
]

print("Number of labels:", len(label_names))
print("Labels:", label_names)
print("Entity types:", entity_types)
print("label2id:", label2id)

Number of labels: 9
Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Entity types: ['PER', 'ORG', 'LOC', 'MISC']
label2id: {'O': 0, 'B-PER': 1, 'I-PER': 2, 'B-ORG': 3, 'I-ORG': 4, 'B-LOC': 5, 'I-LOC': 6, 'B-MISC': 7, 'I-MISC': 8}


In [6]:
# Check the required objects after Runtime Restart

required_objects = [
    "tokenizer",
    "tokenized_dataset",
]

for name in required_objects:
    print(f"{name}:", "Available" if name in globals() else "Missing")

if "tokenizer" in globals():
    print("Tokenizer:", tokenizer.name_or_path)

if "tokenized_dataset" in globals():
    print("Train size:", len(tokenized_dataset["train"]))
    print("Validation size:", len(tokenized_dataset["validation"]))
    print("Test size:", len(tokenized_dataset["test"]))

tokenizer: Missing
tokenized_dataset: Missing


In [7]:
from datasets import load_dataset
from transformers import AutoTokenizer

# ============================================================
# 1. Load CoNLL-2003
# ============================================================

dataset = load_dataset("lhoestq/conll2003")

print("Dataset loaded:")
print("Train:", len(dataset["train"]))
print("Validation:", len(dataset["validation"]))
print("Test:", len(dataset["test"]))


# ============================================================
# 2. Load Tokenizer
# ============================================================

MODEL_CHECKPOINT = "distilbert/distilbert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_CHECKPOINT,
    use_fast=True,
)

print("\nTokenizer:", tokenizer.name_or_path)
print("Fast tokenizer:", tokenizer.is_fast)


# ============================================================
# 3. Tokenize and Align BIO Labels
# ============================================================

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=256,
        is_split_into_words=True,
    )

    all_labels = []

    for batch_index, labels in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)

        aligned_labels = []
        previous_word_id = None

        for word_id in word_ids:

            # Special tokens such as [CLS] and [SEP]
            if word_id is None:
                aligned_labels.append(-100)

            # First subword of a word
            elif word_id != previous_word_id:
                aligned_labels.append(labels[word_id])

            # Additional subwords
            else:
                original_label = labels[word_id]

                # Convert B-X to I-X for continuation subwords
                if original_label in [1, 3, 5, 7]:
                    aligned_labels.append(original_label + 1)
                else:
                    aligned_labels.append(original_label)

            previous_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs


# ============================================================
# 4. Apply Tokenization
# ============================================================

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing and aligning labels",
)


# ============================================================
# 5. Verify
# ============================================================

print("\nTokenized dataset:")
print("Train:", len(tokenized_dataset["train"]))
print("Validation:", len(tokenized_dataset["validation"]))
print("Test:", len(tokenized_dataset["test"]))

print("\nFeatures:")
print(tokenized_dataset["train"].features)

print("\nFirst example:")
print("Input IDs:", tokenized_dataset["train"][0]["input_ids"])
print("Labels:", tokenized_dataset["train"][0]["labels"])

Dataset loaded:
Train: 14041
Validation: 3250
Test: 3453

Tokenizer: distilbert/distilbert-base-cased
Fast tokenizer: True


Tokenizing and aligning labels:   0%|          | 0/14041 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/3250 [00:00<?, ? examples/s]

Tokenizing and aligning labels:   0%|          | 0/3453 [00:00<?, ? examples/s]


Tokenized dataset:
Train: 14041
Validation: 3250
Test: 3453

Features:
{'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}

First example:
Input IDs: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
Labels: [-100, 3, 0, 7, 0, 0, 0, 7, 0, 0, 0, -100]


In [8]:
import torch

from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
)

# ============================================================
# Model Configuration
# ============================================================

MODEL_CHECKPOINT = "distilbert/distilbert-base-cased"

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

# ============================================================
# Dynamic Padding
# ============================================================

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)

# ============================================================
# Move Model to GPU
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model.to(device)

# ============================================================
# Model Information
# ============================================================

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Model:", MODEL_CHECKPOINT)
print("Architecture:", model.__class__.__name__)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: distilbert/distilbert-base-cased
Architecture: DistilBertForTokenClassification
Total parameters: 65,197,833
Trainable parameters: 65,197,833
Device: cuda
GPU: Tesla T4
GPU memory: 14.56 GB


In [9]:
import numpy as np


def extract_entities_from_bio(tags):
    """
    Convert BIO tags into entity spans.

    Returns:
        Set of tuples:
        (entity_type, start_index, end_index)
    """
    entities = set()

    start = None
    entity_type = None

    for index, tag in enumerate(tags):

        if tag == "O":

            if start is not None:
                entities.add(
                    (entity_type, start, index - 1)
                )

                start = None
                entity_type = None

        elif tag.startswith("B-"):

            if start is not None:
                entities.add(
                    (entity_type, start, index - 1)
                )

            entity_type = tag[2:]
            start = index

        elif tag.startswith("I-"):

            current_type = tag[2:]

            if start is None:
                # Safely handle an isolated I-tag.
                entity_type = current_type
                start = index

            elif current_type != entity_type:
                # Treat a type change as a new entity.
                entities.add(
                    (entity_type, start, index - 1)
                )

                entity_type = current_type
                start = index

    # Close the final entity.
    if start is not None:
        entities.add(
            (entity_type, start, len(tags) - 1)
        )

    return entities


def compute_entity_metrics(eval_predictions):
    """
    Compute entity-level precision, recall and F1
    from Hugging Face Trainer predictions.
    """

    predictions = eval_predictions.predictions
    labels = eval_predictions.label_ids

    # Some Trainer versions return predictions as a tuple.
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predicted_ids = np.argmax(
        predictions,
        axis=-1
    )

    all_pred_entities = set()
    all_true_entities = set()

    per_type_pred = {
        entity_type: set()
        for entity_type in entity_types
    }

    per_type_true = {
        entity_type: set()
        for entity_type in entity_types
    }

    for pred_sequence, label_sequence in zip(
        predicted_ids,
        labels
    ):

        pred_tags = []
        true_tags = []

        for pred_id, true_id in zip(
            pred_sequence,
            label_sequence
        ):

            # Ignore special tokens and padding.
            if true_id == -100:
                continue

            pred_tags.append(
                id2label[int(pred_id)]
            )

            true_tags.append(
                id2label[int(true_id)]
            )

        pred_entities = extract_entities_from_bio(
            pred_tags
        )

        true_entities = extract_entities_from_bio(
            true_tags
        )

        all_pred_entities.update(
            pred_entities
        )

        all_true_entities.update(
            true_entities
        )

        for entity in pred_entities:
            per_type_pred[entity[0]].add(entity)

        for entity in true_entities:
            per_type_true[entity[0]].add(entity)

    # --------------------------------------------------------
    # Overall metrics
    # --------------------------------------------------------

    true_positive = len(
        all_pred_entities & all_true_entities
    )

    precision = (
        true_positive / len(all_pred_entities)
        if all_pred_entities
        else 0.0
    )

    recall = (
        true_positive / len(all_true_entities)
        if all_true_entities
        else 0.0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0.0
    )

    # --------------------------------------------------------
    # Per-entity metrics
    # --------------------------------------------------------

    metrics = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

    for entity_type in entity_types:

        predicted = per_type_pred[entity_type]
        actual = per_type_true[entity_type]

        tp = len(
            predicted & actual
        )

        type_precision = (
            tp / len(predicted)
            if predicted
            else 0.0
        )

        type_recall = (
            tp / len(actual)
            if actual
            else 0.0
        )

        type_f1 = (
            2 * type_precision * type_recall
            / (type_precision + type_recall)
            if type_precision + type_recall > 0
            else 0.0
        )

        metrics[f"{entity_type}_precision"] = (
            type_precision
        )

        metrics[f"{entity_type}_recall"] = (
            type_recall
        )

        metrics[f"{entity_type}_f1"] = (
            type_f1
        )

    return metrics


print("Entity-level metrics function: READY")

Entity-level metrics function: READY


In [10]:
from transformers import TrainingArguments, Trainer

# ============================================================
# Training Configuration
# ============================================================

OUTPUT_DIR = "./distilbert-conll2003-ner"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Training
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,

    # Keep the best checkpoint based on validation F1
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    # Speed / GPU
    fp16=torch.cuda.is_available(),

    # Reproducibility
    seed=42,

    # Reduce unnecessary checkpoint storage
    save_total_limit=2,

    # Disable external experiment trackers
    report_to="none",

    # Keep output clean
    disable_tqdm=False,
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_entity_metrics,
)

print("Trainer: READY")
print("Epochs:", training_args.num_train_epochs)
print("Train batch size:", training_args.per_device_train_batch_size)
print("Eval batch size:", training_args.per_device_eval_batch_size)
print("Learning rate:", training_args.learning_rate)
print("FP16:", training_args.fp16)
print("Best model metric:", training_args.metric_for_best_model)

Trainer: READY
Epochs: 3
Train batch size: 16
Eval batch size: 32
Learning rate: 2e-05
FP16: True
Best model metric: f1


In [11]:
# ============================================================
# Fine-tune DistilBERT on CoNLL-2003
# ============================================================

train_result = trainer.train()

print("\nTraining completed successfully.")

print(
    f"Training loss: "
    f"{train_result.training_loss:.4f}"
)

print(
    f"Training runtime: "
    f"{train_result.metrics['train_runtime']:.2f} seconds"
)

print(
    f"Samples / second: "
    f"{train_result.metrics['train_samples_per_second']:.2f}"
)

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Per Precision,Per Recall,Per F1,Org Precision,Org Recall,Org F1,Loc Precision,Loc Recall,Loc F1,Misc Precision,Misc Recall,Misc F1
1,0.087342,0.091344,0.949466,0.924314,0.936721,0.997199,0.967391,0.982069,0.932000,0.935743,0.933868,0.949153,0.961373,0.955224,0.881720,0.792271,0.834606
2,0.055611,0.073101,0.969082,0.948912,0.958891,1.000000,0.983696,0.991781,0.959016,0.939759,0.949290,0.982379,0.957082,0.969565,0.910891,0.888889,0.899756
3,0.032062,0.069390,0.967588,0.960265,0.963913,1.000000,0.980978,0.990398,0.952191,0.959839,0.956000,0.978166,0.961373,0.969697,0.918269,0.922705,0.920482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed successfully.
Training loss: 0.0964
Training runtime: 144.77 seconds
Samples / second: 290.97


In [12]:
# ============================================================
# Final Evaluation on the Unseen CoNLL-2003 Test Set
# ============================================================

test_results = trainer.evaluate(
    eval_dataset=tokenized_dataset["test"],
    metric_key_prefix="test",
)

print("\nFinal Test Results")
print("=" * 60)

for key, value in test_results.items():
    if key.startswith("test_") and isinstance(value, (int, float)):
        print(f"{key}: {value:.6f}")

Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Per Precision,Per Recall,Per F1,Org Precision,Org Recall,Org F1,Loc Precision,Loc Recall,Loc F1,Misc Precision,Misc Recall,Misc F1
0.032062,0.174075,3,0.933194,0.935146,0.934169,0.990881,0.996942,0.993902,0.906122,0.948718,0.926931,0.919431,0.937198,0.928230,0.878613,0.808511,0.842105



Final Test Results
test_loss: 0.174075
test_precision: 0.933194
test_recall: 0.935146
test_f1: 0.934169
test_PER_precision: 0.990881
test_PER_recall: 0.996942
test_PER_f1: 0.993902
test_ORG_precision: 0.906122
test_ORG_recall: 0.948718
test_ORG_f1: 0.926931
test_LOC_precision: 0.919431
test_LOC_recall: 0.937198
test_LOC_f1: 0.928230
test_MISC_precision: 0.878613
test_MISC_recall: 0.808511
test_MISC_f1: 0.842105


In [14]:
import numpy as np

# ============================================================
# Word-Level Final Test Evaluation
# ============================================================

print("Running final word-level evaluation...")

# ------------------------------------------------------------
# 1. Get predictions from the best model
# ------------------------------------------------------------

test_prediction = trainer.predict(
    tokenized_dataset["test"]
)

test_logits = test_prediction.predictions

if isinstance(test_logits, tuple):
    test_logits = test_logits[0]

test_pred_ids = np.argmax(
    test_logits,
    axis=-1
)

print("Predictions shape:", test_pred_ids.shape)


# ------------------------------------------------------------
# 2. Convert Hugging Face Dataset columns to regular Python
#    lists before passing them to the tokenizer.
# ------------------------------------------------------------

test_tokens = list(
    dataset["test"]["tokens"]
)

test_gold_labels = list(
    dataset["test"]["ner_tags"]
)

print("Test examples:", len(test_tokens))


# ------------------------------------------------------------
# 3. Tokenize original words again with word alignment
# ------------------------------------------------------------

test_word_tokens = tokenizer(
    test_tokens,
    truncation=True,
    max_length=256,
    is_split_into_words=True,
)

print("Word-level tokenization: complete")


# ------------------------------------------------------------
# 4. Storage for exact word-level entities
# ------------------------------------------------------------

all_pred_entities = set()
all_true_entities = set()

per_type_pred = {
    entity_type: set()
    for entity_type in entity_types
}

per_type_true = {
    entity_type: set()
    for entity_type in entity_types
}


# ------------------------------------------------------------
# 5. Reconstruct word-level BIO predictions
# ------------------------------------------------------------

for example_index in range(len(test_tokens)):

    word_ids = test_word_tokens.word_ids(
        batch_index=example_index
    )

    original_labels = test_gold_labels[example_index]
    predicted_sequence = test_pred_ids[example_index]

    word_predictions = []
    word_gold_labels = []

    previous_word_id = None

    for token_index, word_id in enumerate(word_ids):

        # Skip special tokens.
        if word_id is None:
            continue

        # Use only the first subword prediction
        # for each original word.
        if word_id == previous_word_id:
            continue

        # Safety check.
        if word_id >= len(original_labels):
            continue

        predicted_label_id = int(
            predicted_sequence[token_index]
        )

        word_predictions.append(
            id2label[predicted_label_id]
        )

        word_gold_labels.append(
            id2label[int(original_labels[word_id])]
        )

        previous_word_id = word_id

    # --------------------------------------------------------
    # Extract exact word-level entity spans.
    # --------------------------------------------------------

    predicted_entities = extract_entities_from_bio(
        word_predictions
    )

    true_entities = extract_entities_from_bio(
        word_gold_labels
    )

    # Add example index so identical spans in different
    # documents cannot collide.
    predicted_entities = {
        (
            entity_type,
            start,
            end,
            example_index
        )
        for entity_type, start, end in predicted_entities
    }

    true_entities = {
        (
            entity_type,
            start,
            end,
            example_index
        )
        for entity_type, start, end in true_entities
    }

    all_pred_entities.update(
        predicted_entities
    )

    all_true_entities.update(
        true_entities
    )

    for entity in predicted_entities:
        per_type_pred[entity[0]].add(entity)

    for entity in true_entities:
        per_type_true[entity[0]].add(entity)


# ------------------------------------------------------------
# 6. Overall exact entity-level metrics
# ------------------------------------------------------------

true_positive = len(
    all_pred_entities & all_true_entities
)

precision = (
    true_positive / len(all_pred_entities)
    if all_pred_entities
    else 0.0
)

recall = (
    true_positive / len(all_true_entities)
    if all_true_entities
    else 0.0
)

f1 = (
    2 * precision * recall
    / (precision + recall)
    if precision + recall > 0
    else 0.0
)


# ------------------------------------------------------------
# 7. Per-entity metrics
# ------------------------------------------------------------

word_level_metrics = {}

for entity_type in entity_types:

    predicted = per_type_pred[entity_type]
    actual = per_type_true[entity_type]

    tp = len(
        predicted & actual
    )

    type_precision = (
        tp / len(predicted)
        if predicted
        else 0.0
    )

    type_recall = (
        tp / len(actual)
        if actual
        else 0.0
    )

    type_f1 = (
        2 * type_precision * type_recall
        / (type_precision + type_recall)
        if type_precision + type_recall > 0
        else 0.0
    )

    word_level_metrics[entity_type] = {
        "precision": type_precision,
        "recall": type_recall,
        "f1": type_f1,
        "support": len(actual),
    }


# ------------------------------------------------------------
# 8. Display final results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("WORD-LEVEL FINAL TEST RESULTS")
print("=" * 70)

print(f"Precision: {precision:.6f}")
print(f"Recall:    {recall:.6f}")
print(f"F1:        {f1:.6f}")

print("\nPer-Entity Results")
print("-" * 70)

for entity_type in entity_types:

    metrics = word_level_metrics[entity_type]

    print(
        f"{entity_type:5s} | "
        f"Precision: {metrics['precision']:.6f} | "
        f"Recall: {metrics['recall']:.6f} | "
        f"F1: {metrics['f1']:.6f} | "
        f"Support: {metrics['support']}"
    )

Running final word-level evaluation...


Predictions shape: (3453, 148)
Test examples: 3453
Word-level tokenization: complete

WORD-LEVEL FINAL TEST RESULTS
Precision: 0.879014
Recall:    0.896601
F1:        0.887720

Per-Entity Results
----------------------------------------------------------------------
PER   | Precision: 0.950249 | Recall: 0.944960 | F1: 0.947597 | Support: 1617
ORG   | Precision: 0.842288 | Recall: 0.877784 | F1: 0.859670 | Support: 1661
LOC   | Precision: 0.918380 | Recall: 0.910671 | F1: 0.914509 | Support: 1668
MISC  | Precision: 0.727865 | Recall: 0.796296 | F1: 0.760544 | Support: 702


In [16]:
# Standard BIO entity-level evaluation without external packages

import numpy as np
from collections import defaultdict

def bio_to_spans(labels):
    """
    Convert a BIO label sequence into exact entity spans.
    Each span is represented as (entity_type, start_index, end_index).
    """
    spans = []
    start = None
    entity_type = None

    for i, label in enumerate(labels):
        if label == "O":
            if entity_type is not None:
                spans.append((entity_type, start, i - 1))
                start = None
                entity_type = None

        elif label.startswith("B-"):
            if entity_type is not None:
                spans.append((entity_type, start, i - 1))

            entity_type = label[2:]
            start = i

        elif label.startswith("I-"):
            current_type = label[2:]

            # Strict BIO handling:
            # An I-X without an active X entity starts a new entity.
            if entity_type is None or entity_type != current_type:
                if entity_type is not None:
                    spans.append((entity_type, start, i - 1))

                entity_type = current_type
                start = i

    if entity_type is not None:
        spans.append((entity_type, start, len(labels) - 1))

    return set(spans)


# Tokenize test set at word level
test_tokens = list(dataset["test"]["tokens"])
test_gold_labels = list(dataset["test"]["ner_tags"])

test_word_tokens = tokenizer(
    test_tokens,
    truncation=True,
    max_length=256,
    is_split_into_words=True,
)

all_true_sequences = []
all_pred_sequences = []

for example_index in range(len(test_tokens)):

    word_ids = test_word_tokens.word_ids(example_index)

    # Model predictions
    input_ids = torch.tensor(
        [test_word_tokens["input_ids"][example_index]],
        device=device
    )

    attention_mask = torch.tensor(
        [test_word_tokens["attention_mask"][example_index]],
        device=device
    )

    with torch.no_grad():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

    predicted_ids = outputs.logits.argmax(dim=-1)[0].cpu().numpy()

    gold_word_labels = []
    predicted_word_labels = []

    previous_word_id = None

    for token_position, word_id in enumerate(word_ids):

        if word_id is None:
            continue

        # Use only the first subword of each original word
        if word_id != previous_word_id:

            gold_label_id = test_gold_labels[example_index][word_id]
            predicted_label_id = int(predicted_ids[token_position])

            gold_word_labels.append(
                id2label[gold_label_id]
            )

            predicted_word_labels.append(
                id2label[predicted_label_id]
            )

        previous_word_id = word_id

    all_true_sequences.append(gold_word_labels)
    all_pred_sequences.append(predicted_word_labels)


# Calculate exact entity-level metrics
true_positive = 0
false_positive = 0
false_negative = 0

per_type = defaultdict(lambda: {
    "tp": 0,
    "fp": 0,
    "fn": 0
})

for true_labels, predicted_labels in zip(
    all_true_sequences,
    all_pred_sequences
):

    true_spans = bio_to_spans(true_labels)
    predicted_spans = bio_to_spans(predicted_labels)

    matched = true_spans & predicted_spans

    true_positive += len(matched)
    false_positive += len(predicted_spans - true_spans)
    false_negative += len(true_spans - predicted_spans)

    for entity in matched:
        per_type[entity[0]]["tp"] += 1

    for entity in predicted_spans - true_spans:
        per_type[entity[0]]["fp"] += 1

    for entity in true_spans - predicted_spans:
        per_type[entity[0]]["fn"] += 1


precision = (
    true_positive / (true_positive + false_positive)
    if true_positive + false_positive > 0 else 0
)

recall = (
    true_positive / (true_positive + false_negative)
    if true_positive + false_negative > 0 else 0
)

f1 = (
    2 * precision * recall / (precision + recall)
    if precision + recall > 0 else 0
)


print("=" * 70)
print("STANDARD BIO ENTITY-LEVEL EVALUATION")
print("=" * 70)

print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")

print("\nPer-Entity Performance")
print("-" * 70)

for entity_type in ["PER", "ORG", "LOC", "MISC"]:

    tp = per_type[entity_type]["tp"]
    fp = per_type[entity_type]["fp"]
    fn = per_type[entity_type]["fn"]

    p = tp / (tp + fp) if tp + fp > 0 else 0
    r = tp / (tp + fn) if tp + fn > 0 else 0
    entity_f1 = (
        2 * p * r / (p + r)
        if p + r > 0 else 0
    )

    support = tp + fn

    print(
        f"{entity_type:5s} | "
        f"P: {p:.4f} | "
        f"R: {r:.4f} | "
        f"F1: {entity_f1:.4f} | "
        f"Support: {support}"
    )

print("=" * 70)

STANDARD BIO ENTITY-LEVEL EVALUATION
Precision : 0.8792
Recall    : 0.8968
F1 Score  : 0.8879

Per-Entity Performance
----------------------------------------------------------------------
PER   | P: 0.9502 | R: 0.9450 | F1: 0.9476 | Support: 1617
ORG   | P: 0.8428 | R: 0.8778 | F1: 0.8599 | Support: 1661
LOC   | P: 0.9184 | R: 0.9107 | F1: 0.9145 | Support: 1668
MISC  | P: 0.7282 | R: 0.7977 | F1: 0.7614 | Support: 702


In [17]:
import spacy
from spacy.pipeline import EntityRuler

# Create a lightweight rule-based NER pipeline
rule_nlp = spacy.blank("en")

ruler = rule_nlp.add_pipe("entity_ruler")

# Generic organization patterns
org_suffixes = [
    "Inc", "Inc.", "Corp", "Corp.", "Corporation",
    "Ltd", "Ltd.", "Limited", "Company", "Co.",
    "Group", "Bank", "University", "Institute",
    "Airlines", "Association", "Foundation",
    "Organization", "Organisation"
]

org_patterns = [
    {
        "label": "ORG",
        "pattern": [
            {"IS_TITLE": True},
            {"LOWER": {"IN": [x.lower() for x in org_suffixes]}}
        ]
    },
    {
        "label": "ORG",
        "pattern": [
            {"IS_UPPER": True, "IS_ALPHA": True, "OP": "+"}
        ]
    }
]

# Person patterns based on common name titles
person_titles = [
    "mr", "mr.", "mrs", "mrs.", "ms", "ms.",
    "dr", "dr.", "prof", "prof.",
    "president", "minister", "senator",
    "governor", "mayor"
]

person_patterns = [
    {
        "label": "PER",
        "pattern": [
            {"LOWER": {"IN": person_titles}},
            {"IS_TITLE": True},
            {"IS_TITLE": True, "OP": "?"}
        ]
    }
]

# Add patterns
ruler.add_patterns(org_patterns + person_patterns)

print("Rule-based NER pipeline created successfully.")
print("Pipeline:", rule_nlp.pipe_names)
print("Patterns:", len(ruler.patterns))

Rule-based NER pipeline created successfully.
Pipeline: ['entity_ruler']
Patterns: 3


In [18]:
from collections import defaultdict

def evaluate_rule_based_ner(dataset_split):
    true_positive = 0
    false_positive = 0
    false_negative = 0

    per_type = defaultdict(lambda: {
        "tp": 0,
        "fp": 0,
        "fn": 0
    })

    for example in dataset_split:

        tokens = example["tokens"]
        gold_ids = example["ner_tags"]

        # Reconstruct the original article
        text = " ".join(tokens)

        # Run rule-based NER
        doc = rule_nlp(text)

        predicted_spans = set()

        for ent in doc.ents:
            # Convert character span back to token indices
            start_token = len(text[:ent.start_char].split())
            end_token = len(text[:ent.end_char].split()) - 1

            if start_token <= end_token:
                predicted_spans.add(
                    (ent.label_, start_token, end_token)
                )

        # Convert gold BIO labels to exact spans
        gold_labels = [
            id2label[label_id]
            for label_id in gold_ids
        ]

        gold_spans = bio_to_spans(gold_labels)

        matched = gold_spans & predicted_spans

        true_positive += len(matched)
        false_positive += len(predicted_spans - gold_spans)
        false_negative += len(gold_spans - predicted_spans)

        for entity in matched:
            per_type[entity[0]]["tp"] += 1

        for entity in predicted_spans - gold_spans:
            per_type[entity[0]]["fp"] += 1

        for entity in gold_spans - predicted_spans:
            per_type[entity[0]]["fn"] += 1

    precision = (
        true_positive / (true_positive + false_positive)
        if true_positive + false_positive > 0 else 0
    )

    recall = (
        true_positive / (true_positive + false_negative)
        if true_positive + false_negative > 0 else 0
    )

    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall > 0 else 0
    )

    return precision, recall, f1, per_type


rule_precision, rule_recall, rule_f1, rule_per_type = (
    evaluate_rule_based_ner(dataset["test"])
)

print("=" * 70)
print("RULE-BASED NER — CO NLL-2003 TEST SET")
print("=" * 70)

print(f"Precision : {rule_precision:.4f}")
print(f"Recall    : {rule_recall:.4f}")
print(f"F1 Score  : {rule_f1:.4f}")

print("\nPer-Entity Performance")
print("-" * 70)

for entity_type in ["PER", "ORG", "LOC", "MISC"]:

    tp = rule_per_type[entity_type]["tp"]
    fp = rule_per_type[entity_type]["fp"]
    fn = rule_per_type[entity_type]["fn"]

    p = tp / (tp + fp) if tp + fp > 0 else 0
    r = tp / (tp + fn) if tp + fn > 0 else 0
    entity_f1 = (
        2 * p * r / (p + r)
        if p + r > 0 else 0
    )

    support = tp + fn

    print(
        f"{entity_type:5s} | "
        f"P: {p:.4f} | "
        f"R: {r:.4f} | "
        f"F1: {entity_f1:.4f} | "
        f"Support: {support}"
    )

print("=" * 70)

RULE-BASED NER — CO NLL-2003 TEST SET
Precision : 0.2610
Recall    : 0.0630
F1 Score  : 0.1015

Per-Entity Performance
----------------------------------------------------------------------
PER   | P: 0.0333 | R: 0.0012 | F1: 0.0024 | Support: 1617
ORG   | P: 0.2715 | R: 0.2131 | F1: 0.2388 | Support: 1661
LOC   | P: 0.0000 | R: 0.0000 | F1: 0.0000 | Support: 1668
MISC  | P: 0.0000 | R: 0.0000 | F1: 0.0000 | Support: 702


In [19]:
import spacy

print("spaCy version:", spacy.__version__)

try:
    md_nlp = spacy.load("en_core_web_md")
    print("en_core_web_md: AVAILABLE")
    print("Pipeline:", md_nlp.pipe_names)
except Exception as e:
    print("en_core_web_md: NOT INSTALLED")
    print(type(e).__name__, ":", str(e)[:300])

spaCy version: 3.8.16
en_core_web_md: NOT INSTALLED
OSError : [E050] Can't find model 'en_core_web_md'. It doesn't seem to be a Python package or a valid path to a data directory.


In [20]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 24.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [21]:
import spacy

md_nlp = spacy.load("en_core_web_md")

print("Model loaded successfully.")
print("Pipeline:", md_nlp.pipe_names)
print("NER labels:", md_nlp.get_pipe("ner").labels)

Model loaded successfully.
Pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
NER labels: ('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


In [22]:
# Evaluate spaCy en_core_web_md on the CoNLL-2003 test set

md_true_positive = 0
md_false_positive = 0
md_false_negative = 0

md_per_type = defaultdict(lambda: {
    "tp": 0,
    "fp": 0,
    "fn": 0
})

# Mapping spaCy entity labels to CoNLL labels
SPACY_TO_CONLL = {
    "PERSON": "PER",
    "ORG": "ORG",
    "GPE": "LOC",
    "LOC": "LOC",
}

for example in dataset["test"]:

    tokens = example["tokens"]
    gold_ids = example["ner_tags"]

    text = " ".join(tokens)

    # spaCy Medium inference
    doc = md_nlp(text)

    predicted_spans = set()

    for ent in doc.ents:

        # Keep only labels compatible with our CoNLL categories
        if ent.label_ not in SPACY_TO_CONLL:
            continue

        entity_type = SPACY_TO_CONLL[ent.label_]

        start_token = len(text[:ent.start_char].split())
        end_token = len(text[:ent.end_char].split()) - 1

        if start_token <= end_token:
            predicted_spans.add(
                (entity_type, start_token, end_token)
            )

    # Gold BIO labels
    gold_labels = [
        id2label[label_id]
        for label_id in gold_ids
    ]

    gold_spans = bio_to_spans(gold_labels)

    matched = gold_spans & predicted_spans

    md_true_positive += len(matched)
    md_false_positive += len(predicted_spans - gold_spans)
    md_false_negative += len(gold_spans - predicted_spans)

    for entity in matched:
        md_per_type[entity[0]]["tp"] += 1

    for entity in predicted_spans - gold_spans:
        md_per_type[entity[0]]["fp"] += 1

    for entity in gold_spans - predicted_spans:
        md_per_type[entity[0]]["fn"] += 1


md_precision = (
    md_true_positive / (md_true_positive + md_false_positive)
    if md_true_positive + md_false_positive > 0 else 0
)

md_recall = (
    md_true_positive / (md_true_positive + md_false_negative)
    if md_true_positive + md_false_negative > 0 else 0
)

md_f1 = (
    2 * md_precision * md_recall / (md_precision + md_recall)
    if md_precision + md_recall > 0 else 0
)


print("=" * 70)
print("spaCy en_core_web_md — CoNLL-2003 TEST SET")
print("=" * 70)

print(f"Precision : {md_precision:.4f}")
print(f"Recall    : {md_recall:.4f}")
print(f"F1 Score  : {md_f1:.4f}")

print("\nPer-Entity Performance")
print("-" * 70)

for entity_type in ["PER", "ORG", "LOC", "MISC"]:

    tp = md_per_type[entity_type]["tp"]
    fp = md_per_type[entity_type]["fp"]
    fn = md_per_type[entity_type]["fn"]

    p = tp / (tp + fp) if tp + fp > 0 else 0
    r = tp / (tp + fn) if tp + fn > 0 else 0

    entity_f1 = (
        2 * p * r / (p + r)
        if p + r > 0 else 0
    )

    support = tp + fn

    print(
        f"{entity_type:5s} | "
        f"P: {p:.4f} | "
        f"R: {r:.4f} | "
        f"F1: {entity_f1:.4f} | "
        f"Support: {support}"
    )

print("=" * 70)

spaCy en_core_web_md — CoNLL-2003 TEST SET
Precision : 0.6451
Recall    : 0.5136
F1 Score  : 0.5719

Per-Entity Performance
----------------------------------------------------------------------
PER   | P: 0.7723 | R: 0.6586 | F1: 0.7109 | Support: 1617
ORG   | P: 0.4487 | R: 0.3293 | F1: 0.3799 | Support: 1661
LOC   | P: 0.6788 | R: 0.7728 | F1: 0.7227 | Support: 1668
MISC  | P: 0.0000 | R: 0.0000 | F1: 0.0000 | Support: 702


In [23]:
# Save the final fine-tuned DistilBERT NER model

FINAL_MODEL_DIR = "./NewsNER-DistilBERT"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("=" * 70)
print("FINAL MODEL SAVED")
print("=" * 70)
print("Path:", FINAL_MODEL_DIR)

import os

for root, dirs, files in os.walk(FINAL_MODEL_DIR):
    for file in files:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"{file:40s} {size_mb:8.2f} MB")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

FINAL MODEL SAVED
Path: ./NewsNER-DistilBERT
config.json                                  0.00 MB
tokenizer.json                               0.64 MB
model.safetensors                          248.72 MB
training_args.bin                            0.00 MB
tokenizer_config.json                        0.00 MB


In [24]:
# Test the fine-tuned NER model on a new news-style article

from transformers import pipeline

ner_pipeline = pipeline(
    "token-classification",
    model=FINAL_MODEL_DIR,
    tokenizer=FINAL_MODEL_DIR,
    aggregation_strategy="simple",
    device=0 if torch.cuda.is_available() else -1
)

sample_article = """
Apple announced a new artificial intelligence research center in London.
The company said that John Smith, Apple's Vice President of AI, will lead
the new team. Microsoft and Google are also expected to participate in
the European AI initiative.
"""

predictions = ner_pipeline(sample_article)

print("=" * 70)
print("NEWS ARTICLE NER TEST")
print("=" * 70)

for entity in predictions:
    print(
        f"{entity['entity_group']:6s} | "
        f"{entity['word']:25s} | "
        f"Score: {entity['score']:.4f}"
    )

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

NEWS ARTICLE NER TEST
ORG    | Apple                     | Score: 0.9958
LOC    | London                    | Score: 0.9992
PER    | John Smith                | Score: 0.9990
ORG    | Apple                     | Score: 0.9936
ORG    | AI                        | Score: 0.9897
ORG    | Microsoft                 | Score: 0.9940
ORG    | Google                    | Score: 0.9652
MISC   | European AI               | Score: 0.8735


In [25]:
import os
import shutil

PROJECT_NAME = "News-NER-Entity-Recognition"

# Create clean export directory
export_dir = f"./{PROJECT_NAME}"
if os.path.exists(export_dir):
    shutil.rmtree(export_dir)

os.makedirs(export_dir, exist_ok=True)

# 1. Copy the fine-tuned model
model_source = "./NewsNER-DistilBERT"
model_destination = os.path.join(export_dir, "NewsNER-DistilBERT")

shutil.copytree(model_source, model_destination)

# 2. Save the current notebook
notebook_files = [
    file for file in os.listdir("/content")
    if file.endswith(".ipynb")
]

if notebook_files:
    # Prefer the most recently modified notebook
    notebook_files.sort(
        key=lambda file: os.path.getmtime(os.path.join("/content", file)),
        reverse=True
    )

    notebook_source = os.path.join("/content", notebook_files[0])
    notebook_destination = os.path.join(
        export_dir,
        "News-Article-NER-with-SpaCy.ipynb"
    )

    shutil.copy2(notebook_source, notebook_destination)

    print("Notebook copied:", notebook_files[0])
else:
    print("WARNING: No notebook file found in /content")


# 3. Create a project information file
project_info = """News-NER-Entity-Recognition

Project: Named Entity Recognition from News Articles
Dataset: CoNLL-2003
Primary Model: DistilBERT fine-tuned for token classification

Final exact entity-level evaluation:
Precision: 0.8792
Recall: 0.8968
F1: 0.8879

Per-Entity F1:
PER: 0.9476
ORG: 0.8599
LOC: 0.9145
MISC: 0.7614

Model directory:
NewsNER-DistilBERT/
"""

with open(
    os.path.join(export_dir, "PROJECT_INFO.txt"),
    "w",
    encoding="utf-8"
) as f:
    f.write(project_info)


# 4. Create ZIP archive
zip_base = f"./{PROJECT_NAME}"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir=export_dir
)

# 5. Show exported files and sizes
print("\n" + "=" * 70)
print("PROJECT EXPORT COMPLETE")
print("=" * 70)

print("ZIP:", zip_path)

zip_size_mb = os.path.getsize(zip_path) / (1024 * 1024)
print(f"ZIP size: {zip_size_mb:.2f} MB")

print("\nExported files:")
for root, dirs, files in os.walk(export_dir):
    for file in files:
        file_path = os.path.join(root, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        relative_path = os.path.relpath(file_path, export_dir)

        print(f"{relative_path:55s} {size_mb:8.2f} MB")


PROJECT EXPORT COMPLETE
ZIP: /content/News-NER-Entity-Recognition.zip
ZIP size: 230.43 MB

Exported files:
PROJECT_INFO.txt                                            0.00 MB
NewsNER-DistilBERT/config.json                              0.00 MB
NewsNER-DistilBERT/tokenizer.json                           0.64 MB
NewsNER-DistilBERT/model.safetensors                      248.72 MB
NewsNER-DistilBERT/training_args.bin                        0.00 MB
NewsNER-DistilBERT/tokenizer_config.json                    0.00 MB


In [26]:
from google.colab import files

files.download("/content/News-NER-Entity-Recognition.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>